# Классификация новостей Lenta.ru по темам

1. Загрузка корпуса `lenta-ru-news` через `corus`.
2. Подготовка данных: подвыборка, предобработка текста и таргета, ускорение обработки, разбиение `60/20/20`.
3. Dummy baseline.
4. `LogisticRegression` с `CountVectorizer` и `TfidfVectorizer`.
5. Подбор гиперпараметров на кросс валидации.
6. Финальная оценка на test, анализ ошибок и выводы.

Помимо accuracy слежу за `macro F1`. Распределение тем в корпусе сильно перекошено (одна «Россия» занимает больше 20% выборки), и accuracy в таких условиях легко маскирует провалы на редких классах.

In [1]:
import functools
import os
import re
import urllib.request
import warnings
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from corus import load_lenta
from IPython.display import display
from joblib import Parallel, delayed
from pymorphy3 import MorphAnalyzer
from razdel import tokenize
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk")

In [2]:
RANDOM_STATE = 42
SAMPLE_SIZE = 100_000
MIN_TOPIC_COUNT = 5
BENCHMARK_SIZE = 50

ROOT = Path.cwd().parent
RAW_DATA_PATH = ROOT / "data" / "lenta-ru-news.csv.gz"
RAW_DATA_URL = "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"

URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
NUM_RE = re.compile(r"\b\d+(?:[.,]\d+)?\b", re.UNICODE)
SPACE_RE = re.compile(r"\s+", re.UNICODE)
WORD_RE = re.compile(r"^[a-zа-яё]+$", re.IGNORECASE)

np.random.seed(RANDOM_STATE)
PARALLEL_CANDIDATES = min(4, os.cpu_count() or 1)
MORPH = MorphAnalyzer()

## 1. Загрузка набора данных через Corus

Из корпуса мне нужны три поля: `title`, `text`, `topic`.

In [3]:
def ensure_raw_dataset() -> Path:
    RAW_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not RAW_DATA_PATH.exists():
        print(f"Downloading raw dataset to {RAW_DATA_PATH} ...")
        urllib.request.urlretrieve(RAW_DATA_URL, RAW_DATA_PATH)
    return RAW_DATA_PATH


raw_path = ensure_raw_dataset()
first_record = next(load_lenta(raw_path))
print("First raw record title:", first_record.title)
print("First raw record topic:", first_record.topic)

First raw record title: Названы регионы России с самой высокой смертностью от рака
First raw record topic: Россия


Дальше собираю записи в таблицу, оставляю нужные поля и выкидываю пустые строки. Заодно смотрю на размер корпуса и число тем, чтобы понять, с каким распределением классов придется работать.

In [4]:
def load_dataset(path: Path) -> pd.DataFrame:
    rows = []
    for record in load_lenta(path):
        title = (record.title or "").strip()
        text = (record.text or "").strip()
        topic = (record.topic or "").strip()
        if not text or not topic:
            continue
        rows.append({"title": title, "text": text, "topic": topic})

    data = pd.DataFrame(rows)
    data = data.drop_duplicates(subset=["title", "text", "topic"]).reset_index(drop=True)
    return data


dataset = load_dataset(raw_path)
print("Dataset size:", len(dataset))
print("Unique topics before filtering:", dataset["topic"].nunique())
display(dataset[["title", "text", "topic"]].head(3))
display(dataset["topic"].value_counts().rename_axis("topic").reset_index(name="count"))

Dataset size: 738949
Unique topics before filtering: 23


,title,text,topic
0,Названы регионы России с самой высокой смертно...,Вице-премьер по социальным вопросам Татьяна Го...,Россия
1,Австрия не представила доказательств вины росс...,Австрийские правоохранительные органы не предс...,Спорт
2,Обнаружено самое счастливое место на планете,Сотрудники социальной сети Instagram проанализ...,Путешествия


,topic,count
0,Россия,160440
1,Мир,136619
2,Экономика,79527
3,Спорт,64411
4,Культура,53790
5,Бывший СССР,53402
6,Наука и техника,53135
7,Интернет и СМИ,44663
8,Из жизни,27604
9,Дом,21730


## 2. Подготовка данных

### 2.1 Репрезентативная подвыборка размера 100 000

Полный корпус содержит почти 739 тысяч новостей. Беру 100 тысяч со стратификацией по `topic`, чтобы пропорции тем в подвыборке повторяли пропорции в оригинале.

Перед выборкой фильтрую слишком редкие темы. Константа `MIN_TOPIC_COUNT = 5` задает минимальное число примеров класса, при котором стратифицированное разбиение `60/20/20` работает корректно.

In [5]:
topic_counts = dataset["topic"].value_counts()
valid_topics_for_splits = topic_counts[topic_counts >= MIN_TOPIC_COUNT].index
filtered_dataset = dataset.loc[dataset["topic"].isin(valid_topics_for_splits)].reset_index(drop=True)

sample_fraction = SAMPLE_SIZE / len(filtered_dataset)
min_count_for_sample = int(np.ceil(MIN_TOPIC_COUNT / sample_fraction))
valid_topics_for_sample = filtered_dataset["topic"].value_counts().loc[lambda s: s >= min_count_for_sample].index
filtered_dataset = filtered_dataset.loc[filtered_dataset["topic"].isin(valid_topics_for_sample)].reset_index(drop=True)

sample_df, _ = train_test_split(
    filtered_dataset,
    train_size=SAMPLE_SIZE,
    stratify=filtered_dataset["topic"],
    random_state=RANDOM_STATE,
)
sample_df = sample_df.reset_index(drop=True)

print("Filtered dataset size:", len(filtered_dataset))
print("Sample size:", len(sample_df))
print("Topics in sample:", sample_df["topic"].nunique())
display(sample_df["topic"].value_counts().rename_axis("topic").reset_index(name="count").head(10))

Filtered dataset size: 738942
Sample size: 100000
Topics in sample: 19


,topic,count
0,Россия,21712
1,Мир,18488
2,Экономика,10762
3,Спорт,8717
4,Культура,7279
5,Бывший СССР,7227
6,Наука и техника,7191
7,Интернет и СМИ,6044
8,Из жизни,3736
9,Дом,2941


### 2.2 Предобработка текста и таргета

Пробую два варианта предобработки.

Первый, `simple`, делает минимум: приведение к нижнему регистру, замену URL на маркер `__url__`, замену чисел на маркер `__num__`, схлопывание лишних пробелов. Маркеры нужны, чтобы модель не создавала отдельный признак на каждую уникальную ссылку или число, а работала с самим фактом их наличия в тексте.

Второй, `lemma`, делает всё то же самое и потом лемматизирует через `pymorphy3`. Лемматизация схлопывает словоформы в одну нормальную форму («протестов», «протесту», «протестами» → «протест»), что уменьшает словарь и должно помочь модели обобщать.

Регулярка `WORD_RE = r'^[a-zа-яё]+$'` нужна, чтобы лемматизатор не тратил время на пунктуацию, маркеры и спецсимволы. Если токен не состоит целиком из букв, он возвращается как есть.

In [6]:
def normalize_text(text: str) -> str:
    text = text.lower()
    text = URL_RE.sub(" __url__ ", text)
    text = NUM_RE.sub(" __num__ ", text)
    text = SPACE_RE.sub(" ", text).strip()
    return text


def preprocess_simple(text: str) -> str:
    return normalize_text(text)

In [7]:
def lemmatize_token_uncached(token: str) -> str:
    if WORD_RE.fullmatch(token) is None:
        return token
    return MORPH.parse(token)[0].normal_form


@functools.cache
def lemmatize_token_cached(token: str) -> str:
    return lemmatize_token_uncached(token)


def preprocess_lemma(text: str, *, use_cache: bool = True) -> str:
    normalized = normalize_text(text)
    tokens = [item.text for item in tokenize(normalized)]
    lemma_fn = lemmatize_token_cached if use_cache else lemmatize_token_uncached
    return " ".join(lemma_fn(token) for token in tokens)


def benchmark_preprocessing(texts) -> pd.DataFrame:
    texts = [str(text) for text in texts]
    rows = []

    lemmatize_token_cached.cache_clear()
    start = perf_counter()
    _ = [preprocess_lemma(text, use_cache=False) for text in texts]
    rows.append(
        {
            "mode": "uncached",
            "seconds": perf_counter() - start,
            "cache_info": "cache not used",
        }
    )

    lemmatize_token_cached.cache_clear()
    start = perf_counter()
    _ = [preprocess_lemma(text, use_cache=True) for text in texts]
    rows.append(
        {
            "mode": "cached_first_pass",
            "seconds": perf_counter() - start,
            "cache_info": str(lemmatize_token_cached.cache_info()),
        }
    )

    start = perf_counter()
    _ = [preprocess_lemma(text, use_cache=True) for text in texts]
    rows.append(
        {
            "mode": "cached_second_pass",
            "seconds": perf_counter() - start,
            "cache_info": str(lemmatize_token_cached.cache_info()),
        }
    )

    return pd.DataFrame(rows)

### 2.2.1 Ускоряем обработку

Лемматизация через `pymorphy3` сама по себе медленная. На 50 текстах без оптимизаций уходит больше 0.6 секунды. Использую следующие приемы для ускорения:

`MorphAnalyzer` создается один раз при старте и переиспользуется. Создание анализатора загружает словари, и делать это заново для каждого текста бессмысленно.

Леммы кэшируются на уровне токенов через `@functools.cache`. В новостном корпусе слова повторяются постоянно, поэтому уже на первом проходе по 50 текстам кэш дает ускорение с 0.68 до 0.24 секунд, а на повторном проходе время падает до 0.04 секунды.

Материализую предобработанные тексты в столбцах датафрейма до обучения, а в кросс валидацию подаю уже готовые строки.

Проверка ускорения за счет кэширования:

In [8]:
benchmark_texts = sample_df["text"].head(BENCHMARK_SIZE).tolist()
benchmark_df = benchmark_preprocessing(benchmark_texts)
display(benchmark_df)

,mode,seconds,cache_info
0,uncached,0.632672,cache not used
1,cached_first_pass,0.223458,"CacheInfo(hits=7580, misses=4148, maxsize=None..."
2,cached_second_pass,0.035444,"CacheInfo(hits=19308, misses=4148, maxsize=Non..."


Строю версии корпуса для экспериментов. Обрабатываю `title` и `text` по отдельности, а потом склеиваю.

In [9]:
sample_df["title_only"] = sample_df["title"].str.strip()
sample_df["text_only"] = sample_df["text"].str.strip()

sample_df["title_only_simple"] = sample_df["title_only"].map(preprocess_simple)
sample_df["text_only_simple"] = sample_df["text_only"].map(preprocess_simple)
sample_df["title_only_lemma"] = sample_df["title_only"].map(preprocess_lemma)
sample_df["text_only_lemma"] = sample_df["text_only"].map(preprocess_lemma)

sample_df["title_text_simple"] = (sample_df["title_only_simple"] + " " + sample_df["text_only_simple"]).str.strip()
sample_df["title_text_lemma"] = (sample_df["title_only_lemma"] + " " + sample_df["text_only_lemma"]).str.strip()

print("Lemma cache info after preprocessing:", lemmatize_token_cached.cache_info())

Lemma cache info after preprocessing: CacheInfo(hits=24574837, misses=458359, maxsize=None, currsize=458359)


### 2.3 Разбиение на обучающую, валидационную и тестовую выборки

Делю корпус на `train` (60 000), `valid` (20 000), `test` (20 000) со стратификацией по `topic`. Все 19 тем присутствуют в каждой части.

In [10]:
train_df, holdout_df = train_test_split(
    sample_df,
    test_size=0.4,
    stratify=sample_df["topic"],
    random_state=RANDOM_STATE,
)
valid_df, test_df = train_test_split(
    holdout_df,
    test_size=0.5,
    stratify=holdout_df["topic"],
    random_state=RANDOM_STATE,
)

for frame, name in [(train_df, "train"), (valid_df, "valid"), (test_df, "test")]:
    print(name, len(frame), frame["topic"].nunique())

split_df = pd.DataFrame(
    {
        "split": ["train", "valid", "test"],
        "size": [len(train_df), len(valid_df), len(test_df)],
    }
)
display(split_df)

topic_distribution = (
    pd.concat(
        [
            train_df.assign(split="train"),
            valid_df.assign(split="valid"),
            test_df.assign(split="test"),
        ]
    )
    .groupby(["split", "topic"])
    .size()
    .rename("count")
    .reset_index()
)
display(topic_distribution.head(12))

train 60000 19
valid 20000 19
test 20000 19


,split,size
0,train,60000
1,valid,20000
2,test,20000


,split,topic,count
0,test,69-я параллель,34
1,test,Библиотека,2
2,test,Бизнес,200
3,test,Бывший СССР,1445
4,test,Дом,588
5,test,Из жизни,747
6,test,Интернет и СМИ,1209
7,test,Крым,18
8,test,Культпросвет,9
9,test,Культура,1456


### 2.4 Выбор формата текстового представления

Прежде чем сравнивать векторизаторы, нужно понять, какой текст подавать на вход. Сравниваю четыре варианта: только `text` (с нормализацией или лемматизацией) и `title + text` (с нормализацией или лемматизацией). Для сравнения использую одну и ту же связку `TfidfVectorizer` + `LogisticRegression`, меняется только входной текст.

Параметры пробного пайплайна стоит пояснить, потому что они переиспользуются дальше.

`token_pattern=r'(?u)\b[\wёЁ]+\b'` отличается от стандартного sklearn шаблона тем, что явно включает букву ё. Без этого слова типа «её», «ещё» дробятся на части.

`min_df=5` отсекает слова, встречающиеся менее чем в 5 документах. Это убирает опечатки и уникальные имена собственные, которые не помогают обобщению.

`max_df=0.8` работает как мягкий стоп лист: слова, встречающиеся в 80%+ документов для классификации по теме бесполезны.

`max_features=100_000` ограничивает словарь сотней тысяч самых частотных признаков. Баланс между покрытием лексики и скоростью.

`sublinear_tf=True` заменяет сырую частоту токена на `1 + log(tf)`. Без этого текст, где слово встречается 50 раз, получит в 50 раз больший вес, чем текст с одним упоминанием. Логарифмирование сглаживает эту разницу.

`C=4.0` задает обратную силу регуляризации в `LogisticRegression`. Дефолт в sklearn равен 1.0, но мне показалось разумным дать модели чуть больше свободы на старте, а потом подкрутить при подборе гиперпараметров.

`solver='saga'` хорошо работает на больших разреженных матрицах в мультиклассовых задачах.

Решение принимаю по `macro F1` на validation.

In [11]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }


def build_probe_pipeline():
    return Pipeline(
        [
            (
                "vect",
                TfidfVectorizer(
                    token_pattern=r"(?u)\b[\wёЁ]+\b",
                    ngram_range=(1, 2),
                    min_df=5,
                    max_df=0.8,
                    max_features=100_000,
                    sublinear_tf=True,
                ),
            ),
            (
                "clf",
                LogisticRegression(
                    solver="saga",
                    C=4.0,
                    max_iter=400,
                    tol=1e-3,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

In [12]:
variant_results = []
for column in ["text_only_simple", "text_only_lemma", "title_text_simple", "title_text_lemma"]:
    probe = build_probe_pipeline()
    probe.fit(train_df[column], train_df["topic"])
    metrics = compute_metrics(valid_df["topic"], probe.predict(valid_df[column]))
    metrics["variant"] = column
    variant_results.append(metrics)
    print(column, metrics)

variant_df = pd.DataFrame(variant_results).sort_values("macro_f1", ascending=False)
best_text_column = variant_df.iloc[0]["variant"]
display(variant_df)

text_only_simple {'accuracy': 0.81595, 'macro_f1': 0.5665079984961271, 'weighted_f1': 0.8107647607721132, 'variant': 'text_only_simple'}
text_only_lemma {'accuracy': 0.8231, 'macro_f1': 0.5939947893574846, 'weighted_f1': 0.8187472666427065, 'variant': 'text_only_lemma'}
title_text_simple {'accuracy': 0.8197, 'macro_f1': 0.5765258184737272, 'weighted_f1': 0.8146586051091533, 'variant': 'title_text_simple'}
title_text_lemma {'accuracy': 0.8271, 'macro_f1': 0.5983719277109598, 'weighted_f1': 0.8228331474100378, 'variant': 'title_text_lemma'}


,accuracy,macro_f1,weighted_f1,variant
3,0.82710,0.598372,0.822833,title_text_lemma
1,0.82310,0.593995,0.818747,text_only_lemma
2,0.81970,0.576526,0.814659,title_text_simple
0,0.81595,0.566508,0.810765,text_only_simple


`title_text_lemma` выиграл с macro F1 = 0.598 на validation, опередив `text_only_lemma` (0.594). Заголовок действительно несет тематический сигнал, а лемматизация заметно помогает: без неё `title_text_simple` даёт только 0.577.

Дальше использую только `title_text_lemma`, меняю векторизатор и параметры модели.

## 3. Dummy baseline

Прежде чем смотреть на содержательные модели, фиксирую нижнюю границу. `DummyClassifier(strategy='most_frequent')` всегда предсказывает самый частый класс обучающей выборки.

In [13]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(np.zeros((len(train_df), 1)), train_df["topic"])
dummy_pred = dummy.predict(np.zeros((len(valid_df), 1)))
dummy_metrics = compute_metrics(valid_df["topic"], dummy_pred)
display(pd.DataFrame([{"model": "Dummy most_frequent", **dummy_metrics}]))
print("Most frequent topic in train:", train_df["topic"].mode().iloc[0])

,model,accuracy,macro_f1,weighted_f1
0,Dummy most_frequent,0.2171,0.018776,0.07745


Most frequent topic in train: Россия


## 4. `LogisticRegression` с двумя вариантами векторизации

Теперь основная модель. `LogisticRegression` с двумя способами построения признаков.

`CountVectorizer` считает абсолютные частоты токенов и n грамм.

`TfidfVectorizer` поверх частот накладывает взвешивание: слова, которые встречаются почти везде, получают меньший вес, а более специфичные для темы усиливаются.

Обе модели работают на `title_text_lemma`.

In [14]:
def build_count_pipeline(*, ngram_range=(1, 2), C=4.0):
    return Pipeline(
        [
            (
                "vect",
                CountVectorizer(
                    token_pattern=r"(?u)\b[\wёЁ]+\b",
                    ngram_range=ngram_range,
                    min_df=5,
                    max_df=0.8,
                    max_features=100_000,
                    binary=False,
                ),
            ),
            (
                "clf",
                LogisticRegression(
                    solver="saga",
                    C=C,
                    max_iter=600,
                    tol=1e-3,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def build_tfidf_pipeline(*, ngram_range=(1, 2), C=4.0):
    return Pipeline(
        [
            (
                "vect",
                TfidfVectorizer(
                    token_pattern=r"(?u)\b[\wёЁ]+\b",
                    ngram_range=ngram_range,
                    min_df=5,
                    max_df=0.8,
                    max_features=100_000,
                    sublinear_tf=True,
                    smooth_idf=True,
                ),
            ),
            (
                "clf",
                LogisticRegression(
                    solver="saga",
                    C=C,
                    max_iter=600,
                    tol=1e-3,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

In [15]:
count_baseline = build_count_pipeline(ngram_range=(1, 2), C=4.0)
tfidf_baseline = build_tfidf_pipeline(ngram_range=(1, 2), C=4.0)

count_baseline.fit(train_df[best_text_column], train_df["topic"])
tfidf_baseline.fit(train_df[best_text_column], train_df["topic"])

count_valid_metrics = compute_metrics(valid_df["topic"], count_baseline.predict(valid_df[best_text_column]))
tfidf_valid_metrics = compute_metrics(valid_df["topic"], tfidf_baseline.predict(valid_df[best_text_column]))

baseline_df = pd.DataFrame(
    [
        {"model": "Dummy most_frequent", **dummy_metrics},
        {"model": "CountVectorizer + LogisticRegression", **count_valid_metrics},
        {"model": "TfidfVectorizer + LogisticRegression", **tfidf_valid_metrics},
    ]
).sort_values("macro_f1", ascending=False)

display(baseline_df)

,model,accuracy,macro_f1,weighted_f1
1,CountVectorizer + LogisticRegression,0.81985,0.632547,0.817428
2,TfidfVectorizer + LogisticRegression,0.82710,0.598372,0.822833
0,Dummy most_frequent,0.21710,0.018776,0.077450


Обе модели уверенно обходят dummy (accuracy 0.217, macro F1 = 0.019). Любопытно, что `CountVectorizer` лидирует по macro F1 (0.633 против 0.598 у TF IDF), хотя по accuracy чуть отстает (0.820 против 0.827). Это значит, что CountVectorizer лучше справляется с редкими классами. Но делать окончательный выбор рано: нужно проверить, не зависит ли результат от конкретных настроек.

## 5. Подбор гиперпараметров на кросс валидации

Подбираю два параметра, которые обычно больше всего влияют в текстовой классификации: диапазон n грамм (`(1,1)` vs `(1,2)`) и сила регуляризации `C` (2.0 и 4.0).

Кросс валидация на 2 фолдах, а не на 5 или 10. Причина прагматичная: на 60 тысячах текстов с векторизацией каждый фолд и так занимает приличное время, а 2 фолда уже дают оценку дисперсии. Метрика отбора та же, `macro F1`. Для каждого кандидата дополнительно смотрю результат на validation.

In [16]:
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE)

count_candidates = [
    {"name": "count_uni_C2", "ngram_range": (1, 1), "C": 2.0},
    {"name": "count_uni_C4", "ngram_range": (1, 1), "C": 4.0},
    {"name": "count_bi_C2", "ngram_range": (1, 2), "C": 2.0},
    {"name": "count_bi_C4", "ngram_range": (1, 2), "C": 4.0},
]

tfidf_candidates = [
    {"name": "tfidf_uni_C2", "ngram_range": (1, 1), "C": 2.0},
    {"name": "tfidf_uni_C4", "ngram_range": (1, 1), "C": 4.0},
    {"name": "tfidf_bi_C2", "ngram_range": (1, 2), "C": 2.0},
    {"name": "tfidf_bi_C4", "ngram_range": (1, 2), "C": 4.0},
]


def evaluate_one_candidate(candidate, builder, model_name):
    model = builder(ngram_range=candidate["ngram_range"], C=candidate["C"])
    cv_scores = cross_val_score(
        model,
        train_df[best_text_column],
        train_df["topic"],
        cv=cv,
        scoring="f1_macro",
        n_jobs=1,
    )
    fitted = builder(ngram_range=candidate["ngram_range"], C=candidate["C"])
    fitted.fit(train_df[best_text_column], train_df["topic"])
    valid_metrics = compute_metrics(valid_df["topic"], fitted.predict(valid_df[best_text_column]))
    return {
        "model": model_name,
        "name": candidate["name"],
        "ngram_range": candidate["ngram_range"],
        "C": candidate["C"],
        "cv_macro_f1_mean": cv_scores.mean(),
        "cv_macro_f1_std": cv_scores.std(),
        "valid_macro_f1": valid_metrics["macro_f1"],
        "valid_accuracy": valid_metrics["accuracy"],
    }


def evaluate_candidates(candidates, builder, model_name):
    rows = Parallel(n_jobs=PARALLEL_CANDIDATES, backend="loky")(
        delayed(evaluate_one_candidate)(candidate, builder, model_name) for candidate in candidates
    )
    return pd.DataFrame(rows).sort_values(["valid_macro_f1", "cv_macro_f1_mean"], ascending=False)

In [17]:
count_tuning_df = evaluate_candidates(count_candidates, build_count_pipeline, "Count")
display(count_tuning_df)
best_count_row = count_tuning_df.iloc[0]

,model,name,ngram_range,C,cv_macro_f1_mean,cv_macro_f1_std,valid_macro_f1,valid_accuracy
3,Count,count_bi_C4,"(1, 2)",4.0,0.566372,0.002881,0.632547,0.81985
2,Count,count_bi_C2,"(1, 2)",2.0,0.566439,0.002950,0.632517,0.81985
0,Count,count_uni_C2,"(1, 1)",2.0,0.570207,0.005341,0.631744,0.81350
1,Count,count_uni_C4,"(1, 1)",4.0,0.570114,0.005375,0.631707,0.81340


In [18]:
tfidf_tuning_df = evaluate_candidates(tfidf_candidates, build_tfidf_pipeline, "TF IDF")
display(tfidf_tuning_df)
best_tfidf_row = tfidf_tuning_df.iloc[0]

,model,name,ngram_range,C,cv_macro_f1_mean,cv_macro_f1_std,valid_macro_f1,valid_accuracy
1,TF IDF,tfidf_uni_C4,"(1, 1)",4.0,0.559651,0.004395,0.605286,0.82165
3,TF IDF,tfidf_bi_C4,"(1, 2)",4.0,0.542212,0.002380,0.598372,0.82710
0,TF IDF,tfidf_uni_C2,"(1, 1)",2.0,0.537724,0.002904,0.575533,0.81900
2,TF IDF,tfidf_bi_C2,"(1, 2)",2.0,0.527337,0.002573,0.568859,0.82140


In [19]:
tuning_summary_df = pd.DataFrame(
    [
        {
            "model": "Best Count",
            "cv_macro_f1_mean": best_count_row["cv_macro_f1_mean"],
            "cv_macro_f1_std": best_count_row["cv_macro_f1_std"],
            "valid_macro_f1": best_count_row["valid_macro_f1"],
        },
        {
            "model": "Best TF IDF",
            "cv_macro_f1_mean": best_tfidf_row["cv_macro_f1_mean"],
            "cv_macro_f1_std": best_tfidf_row["cv_macro_f1_std"],
            "valid_macro_f1": best_tfidf_row["valid_macro_f1"],
        },
    ]
).sort_values("valid_macro_f1", ascending=False)

display(tuning_summary_df)

if best_count_row["valid_macro_f1"] >= best_tfidf_row["valid_macro_f1"]:
    selected_model_name = "Count"
    selected_row = best_count_row
    selected_builder = build_count_pipeline
else:
    selected_model_name = "TF IDF"
    selected_row = best_tfidf_row
    selected_builder = build_tfidf_pipeline

print("Selected model family:", selected_model_name)
print("Selected config:", selected_row.to_dict())

,model,cv_macro_f1_mean,cv_macro_f1_std,valid_macro_f1
0,Best Count,0.566372,0.002881,0.632547
1,Best TF IDF,0.559651,0.004395,0.605286


Selected model family: Count
Selected config: {'model': 'Count', 'name': 'count_bi_C4', 'ngram_range': (1, 2), 'C': 4.0, 'cv_macro_f1_mean': 0.5663716076424403, 'cv_macro_f1_std': 0.0028810225460223493, 'valid_macro_f1': 0.632546852197843, 'valid_accuracy': 0.81985}


Итоговый пайплайн выбираю по сочетанию CV score и качества на валидации. CountVectorizer с биграммами и C=4.0 лидирует: macro F1 = 0.633 на validation, CV score = 0.566. TF IDF уступает, хотя при юниграммах и C=4.0 даёт свой лучший результат (valid macro F1 = 0.605).

## 6. Оценка лучшего пайплайна на отложенной выборке

Объединяю `train` и `valid` (80 000 текстов), обучаю финальную модель (CountVectorizer, биграммы, C=4.0) и только теперь смотрю на test. До этого момента test нигде не участвовал: ни в выборе представления текста, ни в подборе параметров.

In [20]:
train_valid_df = pd.concat([train_df, valid_df], ignore_index=True)
final_model = selected_builder(
    ngram_range=selected_row["ngram_range"],
    C=float(selected_row["C"]),
)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", ConvergenceWarning)
    final_model.fit(train_valid_df[best_text_column], train_valid_df["topic"])

final_convergence_warning = any(isinstance(item.message, ConvergenceWarning) for item in caught)
test_pred = final_model.predict(test_df[best_text_column])
test_metrics = compute_metrics(test_df["topic"], test_pred)

display(pd.DataFrame([test_metrics]))
print("Convergence warning on final fit:", final_convergence_warning)

,accuracy,macro_f1,weighted_f1
0,0.8274,0.628409,0.825391


Convergence warning on final fit: False


### 6.1 Анализ ошибок модели

Смотрю, какие классы модель путает чаще всего, через `classification_report`, таблицу самых частых пар ошибок и конкретные примеры неверных предсказаний.

In [21]:
report = classification_report(test_df["topic"], test_pred, output_dict=True, zero_division=0)
class_report_df = pd.DataFrame(report).T
class_report_df = class_report_df.drop(index=["accuracy", "macro avg", "weighted avg"])
class_report_df = class_report_df.sort_values("f1-score")

display(class_report_df[["precision", "recall", "f1-score", "support"]].head(8))
display(class_report_df[["precision", "recall", "f1-score", "support"]].tail(8))

cm = confusion_matrix(test_df["topic"], test_pred, labels=final_model.classes_)
confusion_pairs = []
for i, true_label in enumerate(final_model.classes_):
    for j, pred_label in enumerate(final_model.classes_):
        if i == j or cm[i, j] == 0:
            continue
        confusion_pairs.append((cm[i, j], true_label, pred_label))

confusion_pairs_df = pd.DataFrame(
    sorted(confusion_pairs, reverse=True)[:12],
    columns=["count", "true_topic", "pred_topic"],
)
display(confusion_pairs_df)

proba = final_model.predict_proba(test_df[best_text_column])
errors_mask = test_pred != test_df["topic"]
errors_df = test_df.loc[errors_mask, ["title", "topic"]].copy()
errors_df["pred_topic"] = test_pred[errors_mask]
errors_df["pred_confidence"] = proba.max(axis=1)[errors_mask]
display(errors_df[["topic", "pred_topic", "pred_confidence", "title"]].head(15))

,precision,recall,f1-score,support
Библиотека,0.000000,0.000000,0.000000,2.0
Культпросвет,0.000000,0.000000,0.000000,9.0
Легпром,0.000000,0.000000,0.000000,3.0
Крым,0.833333,0.277778,0.416667,18.0
Бизнес,0.663934,0.405000,0.503106,200.0
69-я параллель,0.823529,0.411765,0.549020,34.0
Силовые структуры,0.694989,0.600753,0.644444,531.0
Из жизни,0.698366,0.629183,0.661972,747.0


,precision,recall,f1-score,support
Мир,0.810358,0.837977,0.823936,3697.0
Бывший СССР,0.840110,0.843599,0.841851,1445.0
Ценности,0.907104,0.790476,0.844784,210.0
Наука и техника,0.851408,0.840751,0.846046,1438.0
Экономика,0.841227,0.866233,0.853547,2153.0
Дом,0.883636,0.826531,0.854130,588.0
Культура,0.876954,0.885989,0.881449,1456.0
Спорт,0.967853,0.966743,0.967298,1744.0


,count,true_topic,pred_topic
0,272,Мир,Россия
1,267,Россия,Мир
2,139,Силовые структуры,Россия
3,122,Из жизни,Мир
4,99,Экономика,Россия
5,94,Бывший СССР,Россия
6,84,Бизнес,Экономика
7,79,Интернет и СМИ,Россия
8,73,Россия,Экономика
9,69,Мир,Из жизни


,topic,pred_topic,pred_confidence,title
40837,Интернет и СМИ,Культура,0.788606,"""МК-Бульвар"" потребовал обанкротить издательст..."
42806,Россия,Наука и техника,0.506144,"Ракета-носитель ""Рокот"" вывела ""Тома"" и ""Джерр..."
87973,Мир,Интернет и СМИ,0.522229,Захарова вспомнила несуществующую цитату Тэтчер
26799,Бывший СССР,Мир,0.620589,Белорусская оппозиция выбрала новых лидеров
22840,Из жизни,Мир,0.932956,В Вашингтоне марихуана входит в десятку самых ...
34025,Наука и техника,Мир,0.476538,Белоруссия приостановила полеты военной авиации
34366,Дом,Экономика,0.591552,"Управляющего проектом ""Москва-Сити"" оценили в ..."
37234,Путешествия,Из жизни,0.660311,Пассажира поезда наказали за засовывание денег...
1006,Россия,Бывший СССР,0.641106,На российско-латвийской границе перехвачена кр...
82387,Культура,Ценности,0.653911,На торги выставили рекордно крупный бриллиант


## Выводы

Итоговое качество на test: accuracy 0.827, macro F1 = 0.628.

Лучшим оказался `CountVectorizer` с биграммами (C=4.0) на лемматизированных текстах с заголовками. Мне кажется, это довольно объяснимо: для новостного корпуса лемматизация сокращает морфологическую вариативность русского языка, а заголовок несет сильный тематический сигнал.

По ошибкам видна закономерная картина. Спорт классифицируется почти идеально (F1 = 0.967), потому что у спортивных новостей свой специфичный словарь. Самые частые путаницы происходят между «Россия» и «Мир» (272 и 267 ошибок в обе стороны). Это ожидаемо: новость о российской политике, комментирующей международные события, может быть отнесена к любой из двух тем. Похожая история с «Силовые структуры» → «Россия» (139 ошибок), где тематика пересекается.

Три самых маленьких класса («Библиотека», «Легпром», «Культпросвет») получили F1 = 0. Модели видимо было не достаточно примеров, чтобы чему-то научиться. Если бы стояла задача их классифицировать, стоило бы либо объединить их с близкими темами, либо сильно увеличить выборку.